In [0]:
pip install dotenv

In [0]:
dbutils.library.restartPython()

In [0]:
import os
from dotenv import load_dotenv

load_dotenv()

# Set up external location

In [0]:
%sql
-- Create external location
CREATE EXTERNAL LOCATION IF NOT EXISTS external_spotify_adls 
URL 'abfss://data@operationalspotifyadls.dfs.core.windows.net/'
WITH (CREDENTIAL spotify_adls_credential)
COMMENT 'external location to adls';

In [0]:
%python
# show current external locations and its permissions
display(spark.sql("SHOW EXTERNAL LOCATIONS"))
display(spark.sql("SHOW GRANTS ON EXTERNAL LOCATION external_spotify_adls"))

In [0]:
# verify connection
display(dbutils.fs.ls('abfss://data@operationalspotifyadls.dfs.core.windows.net/'))

# Set up catalog + schema 

In [0]:
%sql
-- create catalog
CREATE CATALOG IF NOT EXISTS spotify_dev; 
USE CATALOG spotify_dev; 

-- create schema 
CREATE SCHEMA IF NOT EXISTS spotify_dev.00_landing
COMMENT 'Logical container inside a catalog for 00_landing';

CREATE SCHEMA IF NOT EXISTS spotify_dev.01_bronze
COMMENT 'Logical container inside a catalog for 01_bronze';

CREATE SCHEMA IF NOT EXISTS spotify_dev.02_silver
COMMENT 'Logical container inside a catalog for 02_silver';

CREATE SCHEMA IF NOT EXISTS spotify_dev.03_gold
COMMENT 'Logical container inside a catalog for 03_gold';



-- create volume for each layer
-- USE CATALOG spotify_dev;
-- USE SCHEMA 00_landing; 
-- CREATE VOLUME IF NOT EXISTS spotify_operational_data
-- COMMENT 'Volume to contain Spotify raw operational data';

-- CREATE VOLUME IF NOT EXISTS spotify_streaming_history_data
-- COMMENT 'Volume to contain Spotify streaming history data';


In [0]:
LANDING_VOLUME_PATH = os.getenv("BASE_LANDING_VOLUME_PATH")


In [0]:
%sql
USE CATALOG `spotify_dev`;
USE SCHEMA `00_landing`;

-- Register as an external volume 
CREATE EXTERNAL VOLUME spotify_dev.00_landing.spotify_raw_volume
LOCATION 'abfss://data@operationalspotifyadls.dfs.core.windows.net/ext_volume/raw';